# DSCI 552 Homework 3
**Name:** Yilin Chen

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a)

In [2]:
DATA_DIR = Path("../data/AReM")
ACTIVITIES = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]
SERIES_COLS = ["avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]
COLS = ["time"] + SERIES_COLS


def read_arem_csv(path):
    # "_extra" absorbs the trailing comma in some files
    df = pd.read_csv(path, comment="#", header=None, sep=r"[,\s]+", engine="python",
                     names=COLS + ["_extra"])
    return df[COLS]


def dataset_number(path):
    return int(re.search(r"\d+", path.stem).group())


instances = []
for activity in ACTIVITIES:
    files = sorted((DATA_DIR / activity).glob("dataset*.csv"), key=dataset_number)
    for path in files:
        instances.append({"activity": activity, "dataset": dataset_number(path),
                          "data": read_arem_csv(path)})

### (b)

In [3]:
def is_test(activity, dataset):
    if activity in ("bending1", "bending2"):
        return dataset <= 2
    return dataset <= 3


for inst in instances:
    inst["split"] = "test" if is_test(inst["activity"], inst["dataset"]) else "train"

### (c)
#### (c) i.

**c i Answer:**

Common time-domain features include minimum, maximum, range, mean, median, standard deviation, quartiles,skewness, kurtosis, root mean square  and autocorrelation.

#### (c) ii.

In [4]:
FEATURE_FUNCS = {
    "min":       np.min,
    "max":       np.max,
    "mean":      np.mean,
    "median":    np.median,
    "std":       lambda x: np.std(x, ddof=1),
    "1st_quart": lambda x: np.percentile(x, 25),
    "3rd_quart": lambda x: np.percentile(x, 75),
}


def extract_features(data, l=1):
    """Split each time series into l parts and extract features from every part."""
    features = {}
    for k, col in enumerate(SERIES_COLS, start=1):
        segments = np.array_split(data[col].to_numpy(), l)
        for j, seg in enumerate(segments, start=1):
            suffix = f"{k}" if l == 1 else f"{k}_part{j}"
            for name, func in FEATURE_FUNCS.items():
                features[name + suffix] = func(seg)
    return features


def build_feature_table(instances, l=1):
    rows = []
    for inst in instances:
        row = {"activity": inst["activity"], "dataset": inst["dataset"], "split": inst["split"]}
        row.update(extract_features(inst["data"], l))
        rows.append(row)
    table = pd.DataFrame(rows)
    table.index = pd.RangeIndex(1, len(table) + 1, name="Instance")
    return table

In [5]:
features = build_feature_table(instances, l=1)
feature_cols = [c for c in features.columns if c not in ("activity", "dataset", "split")]
features[feature_cols].round(2)

,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
Instance,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.62,40.50,1.48,39.25,42.00,0.0,1.30,0.36,...,2.19,33.00,36.00,0.00,1.92,0.57,0.43,0.58,0.00,1.30
2,38.00,45.67,42.81,42.50,1.44,42.00,43.67,0.0,1.22,0.37,...,2.00,32.00,34.50,0.00,3.11,0.57,0.43,0.60,0.00,1.30
3,35.00,47.40,43.95,44.33,1.56,43.00,45.00,0.0,1.70,0.43,...,2.00,35.36,36.50,0.00,1.79,0.49,0.43,0.51,0.00,0.94
4,33.00,47.75,42.18,43.50,3.67,39.15,45.00,0.0,3.00,0.70,...,3.85,30.46,36.33,0.00,2.18,0.61,0.50,0.52,0.00,1.00
5,33.00,45.75,41.68,41.75,2.24,41.33,42.75,0.0,2.83,0.54,...,2.41,28.46,31.25,0.00,1.79,0.38,0.43,0.39,0.00,0.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,19.50,45.33,33.59,34.25,4.65,30.25,37.00,0.0,14.67,4.58,...,3.28,13.73,18.25,0.00,8.32,3.26,3.11,1.64,2.05,4.32
85,19.75,45.50,34.32,35.25,4.75,31.00,38.00,0.0,13.47,4.46,...,3.12,13.50,17.75,0.00,9.67,3.43,3.20,1.73,2.16,4.56
86,19.50,46.00,34.55,35.25,4.84,31.25,37.81,0.0,12.47,4.37,...,2.82,14.00,17.75,0.00,10.00,3.34,3.08,1.66,2.16,4.34


#### (c) iii.

In [6]:
X = features[feature_cols]
B = 1000
SEED = 552

point_std = X.std(ddof=1)

boot_stds = np.empty((B, X.shape[1]))
for b in range(B):
    sample = X.sample(frac=1, replace=True, random_state=SEED + b)
    boot_stds[b] = sample.std(ddof=1).to_numpy()

ci_table = pd.DataFrame({
    "Feature": feature_cols,
    "Std (point estimate)": point_std.to_numpy(),
    "90% CI lower (5th pct)": np.percentile(boot_stds, 5, axis=0),
    "90% CI upper (95th pct)": np.percentile(boot_stds, 95, axis=0),
})
ci_table.index = pd.RangeIndex(1, len(ci_table) + 1)
ci_table.round(3)

,Feature,Std (point estimate),90% CI lower (5th pct),90% CI upper (95th pct)
1,min1,9.570,8.282,10.768
2,max1,4.394,3.368,5.322
3,mean1,5.336,4.719,5.910
4,median1,5.440,4.803,6.025
5,std1,1.772,1.576,1.947
6,1st_quart1,6.154,5.576,6.675
7,3rd_quart1,5.139,4.359,5.869
8,min2,0.000,0.000,0.000
9,max2,5.063,4.631,5.406
10,mean2,1.574,1.403,1.699


#### (c) iv.

**c iv Answer:**

The three most important features are **mean, standard deviation and maximum**.
- **Mean:** the avg_rss signals depend on the distance between the sensors. Lying, sitting, standing and bending all put the sensors at different distances, so the mean level of each series is the most direct way to tell these activities apart. 
- **Standard deviation:** It measures how much the signal changes during the 120 seconds. In static activities, such as lying, sitting and standing, the body hardly moves, so the signals are stable. However, in walking and cycling, the body keeps moving, so the signals change a lot. So Standard deviation separates moving activities from static ones.
- **Maximum:** It represents short peaks from sudden movements, such as stepping. These peaks are short, so they barely affect the mean, but the maximum keeps them.

## 2. ISLR 3.7.4

**(a)**

The training RSS of the cubic regression will be lower than or equal to that of the linear regression. The cubic model contains the linear model as a special case (β2 = β3 = 0), and least squares minimizes the training RSS. Therefore, the extra term cannot increase training RSS.

**(b)**

We would expect the linear model's test RSS to be lower. The true relationship is linear, so the extra terms of the cubic model do not reduce bias. They may fit the noise in the training data, so the cubic model may has a higher test RSS. 

**(c)**

The training RSS of the cubic regression will be lower than or equal to that of the linear regression. The cubic model is more flexible and contains the linear model. Therefore, no matter what the true relationship is, cubic model can fit the train data better.

**(d)**

There is not enough information to tell which model will have lower test RSS. If the true relationship is close to linear, the linear regression may have the lower test RSS, but If it is far from linear and is well captured by the cubic model, the cubic regression may have the lower test RSS.